**IMPORTS**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle  # Para persistencia

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

print("Librerías de optimización y persistencia cargadas con éxito.")

**CARGA Y CODIFICACIOND DE DATOS**

In [ ]:
# Cargar datos procesados
df = pd.read_csv('../data/processed/bank_processed.csv')

# Convertir variables categóricas a numéricas (One-Hot Encoding)
categorical_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

X = df_encoded.drop(columns=['deposit'])
y = df_encoded['deposit']

# División estratificada (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Datos preparados para la fase de optimización.")

**EXPERIMENTO 1 - AJUSTE DE HIPERPARAMETROS RANDOM FOREST**

In [ ]:
print("Iniciando búsqueda de hiperparámetros para Random Forest (esto puede tardar unos segundos)...")

# Definir la malla de parámetros a probar
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

# Configurar la búsqueda cruzada buscando optimizar el F1-Score
grid_search_rf = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

best_rf = grid_search_rf.best_estimator_
print(f"Mejores parámetros encontrados: {grid_search_rf.best_params_}")

# Evaluar Random Forest Optimizado
y_pred_rf = best_rf.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
print(f"F1-Score del Random Forest Optimizado: {f1_rf:.4f}")

**EXPERIMENTO 2 - MODELO COMPARATIVO**

In [ ]:
print("Entrenando modelo de comparación: XGBoost...")

# Inicializar y entrenar XGBoost
xgb_model = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Evaluar XGBoost
y_pred_xgb = xgb_model.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb)
print(f"F1-Score del Modelo XGBoost: {f1_xgb:.4f}")

**CUADRO COMPARATIVO DE RENDIMIENTO**

In [ ]:
# Crear dataframe con los resultados para visualizar la tabla comparativa
resultados = pd.DataFrame({
    'Modelo': ['Random Forest Optimizado', 'XGBoost (Comparativo)'],
    'F1-Score (Test)': [f1_rf, f1_xgb]
})

print("--- TABLA COMPARATIVA DE RENDIMIENTO ---")
print(resultados.to_string(index=False))

# Decidir automáticamente cuál es el mejor modelo basado en el F1-Score
if f1_rf > f1_xgb:
    best_model = best_rf
    nombre_mejor = "Random Forest"
else:
    best_model = xgb_model
    nombre_mejor = "XGBoost"

print(f"\nEl modelo ganador para producción es: {nombre_mejor}")

**PERSISTENCIA DEL MODELO GANADOR**

In [ ]:
# Definir la ruta de guardado dentro de la carpeta /models creada
ruta_modelo = '../models/modelo_bank_marketing.pkl'

# Guardar el modelo físicamente
with open(ruta_modelo, 'wb') as archivo:
    pickle.dump(best_model, archivo)

print(f"¡Persistencia completada con éxito!")
print(f"El archivo ejecutable del modelo quedó guardado de forma segura en: {ruta_modelo}")